# 07 — Probability calibration

**Objective.** Fit frozen development models, use 2008 only for probability calibration/cross-fitting, and save matched-reference plus label-specific calibrated model artifacts.

**Scientific contract.** This notebook reports no empirical result until it executes successfully against hash-verified inputs. It writes immutable outputs plus a completion manifest. Expected counts are protocol assertions, not substituted observations.

Permitted block: 2008 probability calibration only. The primary matched-label explanation target always uses separately fitted Platt calibration under an identical procedure.

In [ ]:
# Standard CRUX-VC Colab bootstrap. Git dotfiles restore from the Drive project root; tokens never appear in cells.
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

import shutil
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CRUX_Research")
for _dotfile in (".gitconfig", ".git-credentials"):
    if (DRIVE_PROJECT_ROOT / _dotfile).exists():
        shutil.copy(DRIVE_PROJECT_ROOT / _dotfile, Path.home() / _dotfile)
if (Path.home() / ".git-credentials").exists():
    os.chmod(Path.home() / ".git-credentials", 0o600)

REPO_URL = "https://github.com/anasbiswas1/crux-vc"
REPO_ROOT = Path(os.environ.get("CRUX_REPO_ROOT", "/content/drive/MyDrive/CRUX_Research/crux-vc"))
if not (REPO_ROOT / ".cruxvc-root").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} exists but is not a CRUX-VC checkout")
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    import yaml, pandas, sklearn, pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], check=True)

from cruxvc.runtime import bootstrap_notebook
CTX = bootstrap_notebook("07", suffix=None)
P, CFG, PROFILE = CTX.paths, CTX.config, CTX.profile

In [ ]:
import json, joblib
import numpy as np
import pandas as pd
from cruxvc.calibration import CalibratedModel, PlattCalibrator, choose_calibrator, cross_fitted_calibration_predictions
from cruxvc.io import read_table, write_table
from cruxvc.metrics import binary_prediction_metrics
from cruxvc.models import fit_model, predict_positive

features = read_table(P.processed / "features_strict.parquet")
cohort = read_table(P.processed / "cohort_labels.parquet")
split_ids = read_table(P.protocol / "split_ids.parquet")
registry = read_table(P.models / "near_optimal_registry.parquet")
frame = features.merge(cohort, on=["case_id", "company_permalink", "t0"]).merge(split_ids[["case_id", "time_block"]], on="case_id")
development = frame[frame["time_block"].astype(str).eq("development")].copy()
calibration = frame[frame["time_block"].astype(str).eq("probability_calibration")].copy()
feature_columns = [c for c in features.columns if c not in {"case_id", "company_permalink", "t0"}]

In [ ]:
eligible = registry[registry["near_optimal_family"]].copy()
label_specific = eligible.assign(analysis_role="label_specific_near_optimal")
f36 = eligible[eligible["outcome"].eq("F36")].sort_values(["family", "pooled_log_loss", "complexity_rank", "grid_order", "seed"])
references = f36.groupby("family", as_index=False).first()
matched_rows = []
for reference in references.itertuples(index=False):
    for outcome in CFG["outcomes"]["confirmatory"]:
        row = reference._asdict()
        row["outcome"] = outcome
        row["analysis_role"] = "matched_reference"
        matched_rows.append(row)
calibration_jobs = pd.concat([label_specific, pd.DataFrame(matched_rows)], ignore_index=True)
calibration_jobs = calibration_jobs.drop_duplicates(["outcome", "family", "config_id", "seed", "analysis_role"])

In [ ]:
output_rows = []
prediction_parts = []
model_root = P.models / "calibrated"
for job in calibration_jobs.itertuples(index=False):
    parameters = json.loads(job.parameters_json)
    base = fit_model(development, development[job.outcome], feature_columns, job.family, parameters, int(job.seed))
    raw = predict_positive(base, calibration, feature_columns)
    y = calibration[job.outcome].to_numpy(dtype=int)
    platt_oof = cross_fitted_calibration_predictions(raw, y, "platt", seed=int(job.seed))
    platt = PlattCalibrator().fit(raw, y)
    selected_name, selected_calibrator, method_rows = choose_calibrator(raw, y, seed=int(job.seed))
    headline = CalibratedModel(base, platt, tuple(feature_columns))
    sensitivity = CalibratedModel(base, selected_calibrator, tuple(feature_columns))
    artifact_id = f"{job.outcome}__{job.family}__{job.config_id}__seed{job.seed}__{job.analysis_role}"
    headline_path = model_root / f"{artifact_id}__platt.joblib"
    sensitivity_path = model_root / f"{artifact_id}__selected-{selected_name}.joblib"
    headline_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(headline, headline_path)
    joblib.dump(sensitivity, sensitivity_path)
    metrics = binary_prediction_metrics(y, platt_oof)
    output_rows.append({
        "artifact_id": artifact_id, "outcome": job.outcome, "family": job.family,
        "config_id": job.config_id, "seed": int(job.seed), "parameters_json": job.parameters_json,
        "analysis_role": job.analysis_role, "headline_calibrator": "platt", "selected_sensitivity_calibrator": selected_name,
        "calibrated_model_path": str(headline_path), "sensitivity_model_path": str(sensitivity_path),
        **{f"platt_oof_{k}": v for k, v in metrics.items()},
        "method_comparison_json": json.dumps(method_rows, sort_keys=True),
    })
    prediction_parts.append(pd.DataFrame({
        "case_id": calibration["case_id"].to_numpy(), "outcome": job.outcome, "family": job.family,
        "config_id": job.config_id, "seed": int(job.seed), "analysis_role": job.analysis_role,
        "y_true": y, "raw_probability": raw, "platt_oof_probability": platt_oof,
        "platt_fitted_probability": headline.predict_proba(calibration)[:, 1],
    }))

In [ ]:
calibration_registry = pd.DataFrame(output_rows)
calibration_predictions = pd.concat(prediction_parts, ignore_index=True)
registry_path = write_table(calibration_registry, P.models / "calibration_registry.parquet")
predictions_path = write_table(calibration_predictions, P.predictions / "probability_calibration_2008.parquet")
CTX.recorder.complete([registry_path, predictions_path, *map(Path, calibration_registry["calibrated_model_path"]), *map(Path, calibration_registry["sensitivity_model_path"])])
print(calibration_registry.groupby(["analysis_role", "outcome", "family"]).size())